# 02 — Advanced Capabilities: Structured Outputs & Streaming

This notebook demonstrates two production-critical features of local LLM inference:

1. **Structured outputs with Pydantic** — extract typed, validated data from the Contoso
   confidential report using `format=schema` (constrained decoding via Ollama)
2. **Validation** — how structured output enforcement differs from prompt-based JSON requests
3. **Streaming structured outputs** — accumulate streamed tokens then parse at the end
4. **Model comparison** — which local models comply with structured output constraints reliably

> **Privacy guarantee**: the Contoso report never leaves this machine.

In [1]:
from pathlib import Path

from utils.helpers import check_model_available, check_ollama_running

# ── Models ─────────────────────────────────────────────────────────────────
PRIMARY_MODEL = "mistral:7b"
COMPARISON_MODELS = ["mistral:7b", "qwen3.5:9b", "gemma4:e4b"]

# ── Connectivity guard ──────────────────────────────────────────────────────
if not check_ollama_running():
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

if not check_model_available(PRIMARY_MODEL):
    raise RuntimeError(
        f"Model '{PRIMARY_MODEL}' not found. Run: ollama pull {PRIMARY_MODEL}"
    )

print("✓ Ollama is running")
print(f"✓ Primary model: {PRIMARY_MODEL}")

available = [m for m in COMPARISON_MODELS if check_model_available(m)]
missing = [m for m in COMPARISON_MODELS if m not in available]
print(f"\nComparison models available: {available}")
if missing:
    print(f"Missing (will be skipped):   {missing}")

# ── Load confidential document ──────────────────────────────────────────────
REPORT = Path("data/contoso_report_q4_2024.md").read_text()
print(f"\n✓ Contoso report loaded ({len(REPORT):,} chars)")

✓ Ollama is running
✓ Primary model: mistral:7b

Comparison models available: ['mistral:7b', 'qwen3.5:9b', 'gemma4:e4b']

✓ Contoso report loaded (5,585 chars)


## 1. Structured outputs with Pydantic

The standard approach to structured output is to ask the model to "return JSON" in the
system prompt — but this is unreliable: the model may wrap the output in markdown code
fences, add commentary, or produce subtly invalid JSON.

Ollama's `format` parameter uses **constrained decoding**: the model's token sampling is
restricted to tokens that keep the output valid against the provided JSON schema.
This guarantees syntactically valid JSON, not just requested JSON.

```python
response = chat(model=..., messages=..., format=MyModel.model_json_schema())
result   = MyModel.model_validate_json(response.message.content)
```

In [2]:
from pydantic import BaseModel, Field


class AttritionData(BaseModel):
    q4_rate_percent: float = Field(
        description="Q4 2024 attrition rate as a percentage"
    )
    total_departures_q4: int = Field(
        description="Total number of employee departures in Q4 2024"
    )
    top_departure_reason: str = Field(
        description="Most common reason cited in exit interviews"
    )
    engineering_departures: int = Field(
        description="Number of Engineering department departures in Q4"
    )


class ContosoCoreMetrics(BaseModel):
    total_headcount: int = Field(description="Total headcount on 31 Dec 2024")
    average_salary_gbp: int = Field(description="Average base salary in GBP")
    median_salary_gbp: int = Field(description="Median base salary in GBP")
    attrition: AttritionData
    q1_2025_new_hires_target: int = Field(
        description="Target number of new hires for Q1 2025"
    )
    highest_severity_risk: str = Field(
        description="The single highest-severity risk listed in section 8"
    )


print("Schema defined:")
print(ContosoCoreMetrics.model_json_schema())

Schema defined:
{'$defs': {'AttritionData': {'properties': {'q4_rate_percent': {'description': 'Q4 2024 attrition rate as a percentage', 'title': 'Q4 Rate Percent', 'type': 'number'}, 'total_departures_q4': {'description': 'Total number of employee departures in Q4 2024', 'title': 'Total Departures Q4', 'type': 'integer'}, 'top_departure_reason': {'description': 'Most common reason cited in exit interviews', 'title': 'Top Departure Reason', 'type': 'string'}, 'engineering_departures': {'description': 'Number of Engineering department departures in Q4', 'title': 'Engineering Departures', 'type': 'integer'}}, 'required': ['q4_rate_percent', 'total_departures_q4', 'top_departure_reason', 'engineering_departures'], 'title': 'AttritionData', 'type': 'object'}}, 'properties': {'total_headcount': {'description': 'Total headcount on 31 Dec 2024', 'title': 'Total Headcount', 'type': 'integer'}, 'average_salary_gbp': {'description': 'Average base salary in GBP', 'title': 'Average Salary Gbp', 't

In [5]:
import time

from ollama import chat

messages = [
    {
        "role": "system",
        "content": (
            "You are a data extraction assistant. "
            "Extract information from the provided HR report. "
            "Return only the requested fields — no explanation."
        ),
    },
    {
        "role": "user",
        "content": f"Extract the core metrics from this report:\n\n{REPORT}",
    },
]

t0 = time.time()
response = chat(
    model=PRIMARY_MODEL,
    messages=messages,
    format=ContosoCoreMetrics.model_json_schema(),
    options={"temperature": 0},
)
elapsed = time.time() - t0

metrics = ContosoCoreMetrics.model_validate_json(response.message.content)

print(f"Extracted in {elapsed:.1f}s\n")
print(metrics.model_dump_json(indent=2))

Extracted in 4.2s

{
  "total_headcount": 214,
  "average_salary_gbp": 81300,
  "median_salary_gbp": 68500,
  "attrition": {
    "q4_rate_percent": 4.2,
    "total_departures_q4": 9,
    "top_departure_reason": "Better compensation elsewhere",
    "engineering_departures": 6
  },
  "q1_2025_new_hires_target": 24,
  "highest_severity_risk": "Engineering L3/L4 attrition (6 of 9 Q4 departures)"
}


In [6]:
# Ground-truth verification — values are readable directly from the Contoso report
GROUND_TRUTH = {
    "total_headcount": 214,
    "average_salary_gbp": 81300,
    "median_salary_gbp": 68500,
    "attrition.q4_rate_percent": 4.2,
    "attrition.total_departures_q4": 9,
    "attrition.engineering_departures": 4,
    "q1_2025_new_hires_target": 24,
}

extracted = {
    "total_headcount": metrics.total_headcount,
    "average_salary_gbp": metrics.average_salary_gbp,
    "median_salary_gbp": metrics.median_salary_gbp,
    "attrition.q4_rate_percent": metrics.attrition.q4_rate_percent,
    "attrition.total_departures_q4": metrics.attrition.total_departures_q4,
    "attrition.engineering_departures": metrics.attrition.engineering_departures,
    "q1_2025_new_hires_target": metrics.q1_2025_new_hires_target,
}

print(f"{'Field':<45} {'Expected':>10}  {'Extracted':>10}  {'OK?'}")
print("─" * 75)
all_ok = True
for field, expected in GROUND_TRUTH.items():
    got = extracted[field]
    ok = got == expected
    all_ok = all_ok and ok
    mark = "✓" if ok else "✗"
    print(f"{field:<45} {str(expected):>10}  {str(got):>10}  {mark}")

print(f"\n{'All correct ✓' if all_ok else 'Some fields incorrect ✗'}")

Field                                           Expected   Extracted  OK?
───────────────────────────────────────────────────────────────────────────
total_headcount                                      214         214  ✓
average_salary_gbp                                 81300       81300  ✓
median_salary_gbp                                  68500       68500  ✓
attrition.q4_rate_percent                            4.2         4.2  ✓
attrition.total_departures_q4                          9           9  ✓
attrition.engineering_departures                       4           6  ✗
q1_2025_new_hires_target                              24          24  ✓

Some fields incorrect ✗


## 2. Validation — format enforcement vs prompt-only

Without `format=schema`, the model may produce:
- JSON wrapped in ` ```json ` markdown fences
- Extra commentary before or after the JSON
- Subtly wrong types (a float as a string, a missing field)

All of these break `model_validate_json()`. The cell below shows the fragile
"prompt-only" approach and how to handle failures gracefully.

In [7]:
import json

from pydantic import ValidationError


# A simpler schema for this comparison
class AttritionSnapshot(BaseModel):
    q4_rate_percent: float
    total_departures: int
    largest_impacted_department: str


prompt_only_messages = [
    {
        "role": "system",
        "content": (
            "Return ONLY a JSON object with keys: q4_rate_percent (float), "
            "total_departures (int), largest_impacted_department (str). "
            "No markdown, no explanation."
        ),
    },
    {
        "role": "user",
        "content": f"Extract attrition data from:\n\n{REPORT}",
    },
]


def try_parse(raw: str, schema: type[BaseModel]) -> tuple[BaseModel | None, str]:
    """Attempt to parse raw model output; strip markdown fences if present."""
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        cleaned = "\n".join(lines[1:-1] if lines[-1] == "```" else lines[1:])
    try:
        return schema.model_validate_json(cleaned), "ok"
    except (ValidationError, json.JSONDecodeError) as exc:
        return None, str(exc)


# ── Without format enforcement ──────────────────────────────────────────────
raw_response = chat(
    model=PRIMARY_MODEL,
    messages=prompt_only_messages,
    options={"temperature": 0},
)
raw_text = raw_response.message.content

result_no_format, status_no_format = try_parse(raw_text, AttritionSnapshot)
print("── Without format=schema ─────────────────────────")
print(f"Raw output:\n{raw_text}\n")
print(f"Parse status: {status_no_format}")
if result_no_format:
    print(f"Result: {result_no_format}")

# ── With format enforcement ─────────────────────────────────────────────────
format_response = chat(
    model=PRIMARY_MODEL,
    messages=prompt_only_messages,
    format=AttritionSnapshot.model_json_schema(),
    options={"temperature": 0},
)
result_with_format, status_with_format = try_parse(
    format_response.message.content, AttritionSnapshot
)
print("\n── With format=schema ────────────────────────────")
print(f"Raw output:\n{format_response.message.content}\n")
print(f"Parse status: {status_with_format}")
if result_with_format:
    print(f"Result: {result_with_format}")

── Without format=schema ─────────────────────────
Raw output:
 {
  "q4_rate_percent": 4.2,
  "total_departures": 9,
  "largest_impacted_department": "Engineering"
}

Parse status: ok
Result: q4_rate_percent=4.2 total_departures=9 largest_impacted_department='Engineering'

── With format=schema ────────────────────────────
Raw output:
{"q4_rate_percent": 0.042, "total_departures": 9, "largest_impacted_department": "Engineering"}

Parse status: ok
Result: q4_rate_percent=0.042 total_departures=9 largest_impacted_department='Engineering'


## 3. Streaming structured outputs

Streaming and structured outputs can be combined: set both `stream=True` and
`format=schema`. Each chunk is a partial JSON token — you accumulate them and parse
the complete object only once `chunk.done` is `True`.

Use case: show a live progress indicator while a long extraction is running, then
surface the parsed result the moment the stream ends.

In [8]:
accumulated = ""
final_chunk = None

print("Streaming JSON tokens:\n")

stream = chat(
    model=PRIMARY_MODEL,
    messages=messages,  # same extraction prompt as section 1
    format=ContosoCoreMetrics.model_json_schema(),
    options={"temperature": 0},
    stream=True,
)

for chunk in stream:
    token = chunk.message.content
    if token:
        accumulated += token
        print(token, end="", flush=True)
    if chunk.done:
        final_chunk = chunk

# Parse only after the stream is complete
streamed_metrics = ContosoCoreMetrics.model_validate_json(accumulated)

speed = final_chunk.eval_count / (final_chunk.eval_duration / 1e9)
print("\n\n── Parse result ──────────────────────────────────")
print(f"headcount={streamed_metrics.total_headcount}, "
      f"attrition={streamed_metrics.attrition.q4_rate_percent}%, "
      f"new_hires={streamed_metrics.q1_2025_new_hires_target}")
print(f"Speed: {speed:.1f} tokens/sec | "
      f"Tokens: {final_chunk.eval_count}")

Streaming JSON tokens:

{
"total_headcount": 214,
"average_salary_gbp": 81300,
"median_salary_gbp": 68500,
"attrition": {
    "q4_rate_percent": 4.2,
    "total_departures_q4": 9,
    "top_departure_reason": "Better compensation elsewhere",
    "engineering_departures": 6
},
"q1_2025_new_hires_target": 24,
"highest_severity_risk": "Engineering L3/L4 attrition (6 of 9 Q4 departures)"
}

── Parse result ──────────────────────────────────
headcount=214, attrition=4.2%, new_hires=24
Speed: 45.2 tokens/sec | Tokens: 176


## 4. Model comparison — structured output compliance

Same extraction task (`AttritionSnapshot`) sent to all available local models.
Each model is tested with `format=schema`. We record:

- Whether Pydantic validation passes
- Factual accuracy of the extracted values vs ground truth
- Generation speed

In [9]:
EXPECTED = AttritionSnapshot(
    q4_rate_percent=4.2,
    total_departures=9,
    largest_impacted_department="Engineering",
)

results = []

for model in available:
    t0 = time.time()
    try:
        resp = chat(
            model=model,
            messages=prompt_only_messages,
            format=AttritionSnapshot.model_json_schema(),
            options={"temperature": 0},
        )
        elapsed = time.time() - t0
        parsed = AttritionSnapshot.model_validate_json(resp.message.content)
        accurate = (
            parsed.q4_rate_percent == EXPECTED.q4_rate_percent
            and parsed.total_departures == EXPECTED.total_departures
        )
        speed = resp.eval_count / (resp.eval_duration / 1e9)
        results.append({
            "model": model,
            "valid_json": True,
            "accurate": accurate,
            "q4_rate": parsed.q4_rate_percent,
            "departures": parsed.total_departures,
            "department": parsed.largest_impacted_department,
            "speed_tps": round(speed, 1),
            "elapsed_s": round(elapsed, 1),
        })
    except (ValidationError, json.JSONDecodeError, Exception) as exc:
        elapsed = time.time() - t0
        results.append({
            "model": model,
            "valid_json": False,
            "accurate": False,
            "error": str(exc)[:60],
            "elapsed_s": round(elapsed, 1),
        })

# ── Display table ───────────────────────────────────────────────────────────
print(f"{'Model':<20} {'Valid':>6}  {'Accurate':>8}  "
      f"{'Rate%':>6}  {'Dept':>3}  {'Speed':>8}")
print("─" * 70)
for r in results:
    if r["valid_json"]:
        acc = "✓" if r["accurate"] else "✗"
        print(
            f"{r['model']:<20} {'✓':>6}  {acc:>8}  "
            f"{r['q4_rate']:>6}  {r['departures']:>3}  "
            f"{r['speed_tps']:>6.1f} t/s"
        )
    else:
        print(f"{r['model']:<20} {'✗':>6}  {'—':>8}  {'—':>6}  {'—':>3}  —")

print(f"\nExpected: rate={EXPECTED.q4_rate_percent}%, "
      f"departures={EXPECTED.total_departures}, "
      f"dept={EXPECTED.largest_impacted_department}")

Model                 Valid  Accurate   Rate%  Dept     Speed
──────────────────────────────────────────────────────────────────────
mistral:7b                ✓         ✗   0.042    9    46.6 t/s
qwen3.5:9b                ✓         ✓     4.2    9    25.4 t/s
gemma4:e4b                ✓         ✓     4.2    9    54.8 t/s

Expected: rate=4.2%, departures=9, dept=Engineering
